# handlers

> Tier 1 handler functions for queue reorder, remove, and clear operations.

In [ ]:
#| default_exp handlers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Optional

from cjm_fasthtml_sortable_queue.config import SortableQueueConfig, get_item_key
from cjm_fasthtml_sortable_queue.html_ids import SortableQueueHtmlIds
from cjm_fasthtml_sortable_queue.models import SortableQueueUrls, reorder_by_keys, reorder_by_direction
from cjm_fasthtml_sortable_queue.rendering import render_sortable_queue

In [ ]:
#| export
def handle_reorder(
    config: SortableQueueConfig,  # Queue configuration
    ids: SortableQueueHtmlIds,  # HTML ID generators
    urls: SortableQueueUrls,  # URL endpoints
    items: List[dict],  # Current item list
    new_key_order: List[str],  # Keys in new order (from form_data.getlist("item"))
    render_content: Callable[[dict, int], Any],  # Custom content callback
    **render_kwargs,  # Additional kwargs passed to render_sortable_queue
) -> tuple:  # (reordered_items, rendered_panel)
    """Reorder items by key order and return the updated list and rendered panel."""
    key_fn = lambda item: get_item_key(config, item)
    reordered = reorder_by_keys(items, new_key_order, key_fn)
    panel = render_sortable_queue(config, ids, urls, reordered, render_content, **render_kwargs)
    return reordered, panel

In [ ]:
#| export
def handle_reorder_by_direction(
    config: SortableQueueConfig,  # Queue configuration
    ids: SortableQueueHtmlIds,  # HTML ID generators
    urls: SortableQueueUrls,  # URL endpoints
    items: List[dict],  # Current item list
    item_key: str,  # Key of item to move
    direction: str,  # "up" or "down"
    render_content: Callable[[dict, int], Any],  # Custom content callback
    **render_kwargs,  # Additional kwargs passed to render_sortable_queue
) -> tuple:  # (reordered_items, rendered_panel)
    """Move an item up or down and return the updated list and rendered panel."""
    key_fn = lambda item: get_item_key(config, item)
    reordered = reorder_by_direction(items, item_key, direction, key_fn)
    panel = render_sortable_queue(config, ids, urls, reordered, render_content, **render_kwargs)
    return reordered, panel

In [ ]:
#| export
def handle_remove(
    config: SortableQueueConfig,  # Queue configuration
    ids: SortableQueueHtmlIds,  # HTML ID generators
    urls: SortableQueueUrls,  # URL endpoints
    items: List[dict],  # Current item list
    remove_key: str,  # Key of item to remove
    render_content: Callable[[dict, int], Any],  # Custom content callback
    **render_kwargs,  # Additional kwargs passed to render_sortable_queue
) -> tuple:  # (updated_items, rendered_panel)
    """Remove an item by key and return the updated list and rendered panel."""
    key_fn = lambda item: get_item_key(config, item)
    updated = [item for item in items if key_fn(item) != remove_key]
    panel = render_sortable_queue(config, ids, urls, updated, render_content, **render_kwargs)
    return updated, panel

In [ ]:
#| export
def handle_clear(
    config: SortableQueueConfig,  # Queue configuration
    ids: SortableQueueHtmlIds,  # HTML ID generators
    urls: SortableQueueUrls,  # URL endpoints
    render_content: Callable[[dict, int], Any],  # Custom content callback
    **render_kwargs,  # Additional kwargs passed to render_sortable_queue
) -> tuple:  # (empty_list, rendered_panel)
    """Clear all items and return an empty list and rendered panel."""
    panel = render_sortable_queue(config, ids, urls, [], render_content, **render_kwargs)
    return [], panel

## Tests

In [ ]:
from fasthtml.common import Span, to_xml

# Test setup
config = SortableQueueConfig(prefix="th")
ids_obj = SortableQueueHtmlIds(prefix="th")
urls = SortableQueueUrls(reorder="/q/reorder", remove="/q/remove", clear="/q/clear")
test_items = [{"id": "a", "name": "Alpha"}, {"id": "b", "name": "Beta"}, {"id": "c", "name": "Charlie"}]

def content_fn(item, index):
    return Span(item["name"])

# --- handle_reorder ---
reordered, panel = handle_reorder(config, ids_obj, urls, test_items, ["c", "a", "b"], content_fn)
assert [i["id"] for i in reordered] == ["c", "a", "b"]
xml = to_xml(panel)
assert "Charlie" in xml  # Items rendered

# --- handle_reorder_by_direction ---
moved, panel = handle_reorder_by_direction(config, ids_obj, urls, test_items, "b", "up", content_fn)
assert [i["id"] for i in moved] == ["b", "a", "c"]
xml = to_xml(panel)
assert 'id="th-queue-container"' in xml

# --- handle_remove ---
updated, panel = handle_remove(config, ids_obj, urls, test_items, "b", content_fn)
assert [i["id"] for i in updated] == ["a", "c"]
xml = to_xml(panel)
assert "Beta" not in xml
assert "Alpha" in xml and "Charlie" in xml

# Remove last item — should show empty state
updated2, panel2 = handle_remove(config, ids_obj, urls, [{"id": "x", "name": "Only"}], "x", content_fn)
assert updated2 == []
assert "No items selected" in to_xml(panel2)

# --- handle_clear ---
cleared, panel = handle_clear(config, ids_obj, urls, content_fn)
assert cleared == []
xml = to_xml(panel)
assert "No items selected" in xml
assert "th-queue-list" not in xml

print("All handler tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()